In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join('..'))) 
import sys
from dotenv import load_dotenv
import numpy as np
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
import pandas as pd
from src.utils.main_utils import get_spark_session, read_yaml_file, calculate_cost
from src.exception import MyException

In [ ]:
load_dotenv()
def clean_holdout(df, drop_cols):
    """Mirrors data_cleaning_component exactly"""
    from pyspark.sql import functions as F
    from pyspark.sql.types import IntegerType, FloatType

    df = df.withColumn("Transaction Date", F.to_timestamp("Transaction Date"))
    df = df.withColumn("Transaction Day", F.dayofmonth("Transaction Date"))
    df = df.withColumn("Transaction DOW", (F.dayofweek("Transaction Date") + 5) % 7)
    df = df.withColumn("Transaction Month", F.month("Transaction Date"))

    mean_row = df.select(F.round(F.mean("Customer Age"), 0).alias("m")).collect()[0]
    mean_age = float(mean_row["m"])
    df = df.withColumn(
        "Customer Age",
        F.when(F.col("Customer Age") <= -9, F.abs(F.col("Customer Age"))).otherwise(F.col("Customer Age")),
    )
    df = df.withColumn(
        "Customer Age",
        F.when(F.col("Customer Age") < 9, F.lit(mean_age)).otherwise(F.col("Customer Age")),
    )
    df = df.withColumn(
        "Is Address Match", (F.col("Shipping Address") == F.col("Billing Address")).cast("int")
    )
    df = df.drop(*[c for c in drop_cols if c in df.columns])

    for field in df.schema.fields:
        type_name = field.dataType.typeName()
        if type_name in ("long", "short", "byte"):
            df = df.withColumn(field.name, F.col(field.name).cast(IntegerType()))
        elif type_name == "double":
            df = df.withColumn(field.name, F.col(field.name).cast(FloatType()))
    return df

In [15]:
def main():
    spark = None
    try:
        import dagshub
        dagshub.auth.add_app_token(os.environ["DAGSHUB_USER_TOKEN"])
        dagshub.init(
            repo_owner=os.environ["DAGSHUB_REPO_OWNER"], repo_name=os.environ["DAGSHUB_REPO_NAME"], mlflow=True
        )

        client = MlflowClient()
        registered_model_name = "fraud_detection_model"
        repo_name = os.environ["LAKEFS_REPO_NAME"]

        schema = read_yaml_file("../config/schema.yaml")
        target_col = schema["target_column"]
        drop_cols = schema["drop_columns"]

        spark = get_spark_session("holdout-evaluation")

        # --- Resolve champion: model, threshold, preprocessing pipeline ---
        champion_version = client.get_model_version_by_alias(registered_model_name, "champion")
        model = mlflow.xgboost.load_model(f"models:/{registered_model_name}@champion")
        threshold = float(champion_version.tags.get("threshold", 0.75))

        champion_run = client.get_run(champion_version.run_id)
        parent_run_id = champion_run.data.tags["mlflow.parentRunId"]
        preprocessing_pipeline = mlflow.spark.load_model(f"runs:/{parent_run_id}/pipeline_model")

        import io
        import pandas as pd
        import lakefs as lakefs_sdk

        repo = lakefs_sdk.Repository(repo_name)
        obj = repo.branch("main").object("holdout/eval.parquet")
        with obj.reader(pre_sign=False) as f:
            holdout_pdf = pd.read_parquet(io.BytesIO(f.read()))
        holdout_pdf["Transaction Date"] = holdout_pdf["Transaction Date"].astype("datetime64[us]")

        raw_df = spark.createDataFrame(holdout_pdf)

        clean_df = clean_holdout(raw_df, drop_cols)
        transformed_df = preprocessing_pipeline.transform(clean_df).select("features", target_col)

        pdf = transformed_df.toPandas()
        X = np.array(pdf["features"].apply(lambda v: v.toArray()).tolist())
        y_true = pdf[target_col].values

        spark.stop()
        spark = None

        # --- Score + apply the champion's own tuned threshold ---
        y_probs = model.predict_proba(X)[:, 1]
        y_pred = (y_probs >= threshold).astype(int)

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fn_rate, fp_rate, cost = calculate_cost(tn, fp, fn, tp)

        print("\n=== TRUE HOLDOUT EVALUATION (full set, batch-scored) ===")
        print(f"champion version: {champion_version.version}")
        print(f"threshold:        {threshold}")
        print(f"rows evaluated:   {len(y_true)}")
        print(f"accuracy:         {accuracy:.4f}")
        print(f"precision:        {precision:.4f}")
        print(f"recall:           {recall:.4f}")
        print(f"fn_rate:          {fn_rate:.4f}")
        print(f"fp_rate:          {fp_rate:.4f}")
        print(f"cost:             {cost:.4f}")
    except Exception as e:
        raise MyException(e, sys)
    finally:
        if spark is not None:
            spark.stop()

In [16]:
main()

Initialized MLflow to track repo "Tchaikovsky29/Fraud-Detection"

Repository Tchaikovsky29/Fraud-Detection initialized!

2026/09/05 01:14:51 INFO mlflow.spark: URI 'runs:/bf75c720e34e49bb9e9b261caf294b5e/pipeline_model/sparkml' does not point to the current DFS.
2026/09/05 01:14:51 INFO mlflow.spark: File 'runs:/bf75c720e34e49bb9e9b261caf294b5e/pipeline_model/sparkml' not found on DFS. Will attempt to upload the file.



=== TRUE HOLDOUT EVALUATION (full set, batch-scored) ===
champion version: 4
threshold:        0.75
rows evaluated:   23634
accuracy:         0.9350
precision:        0.3846
recall:           0.4280
fn_rate:          0.5720
fp_rate:          0.0373
cost:             0.1833
